In [2]:
import numpy as np 
from spread_spectrum import SpreadSpectrum

In [3]:
ss = SpreadSpectrum(8, 1024, 0)
ss = SpreadSpectrum(16, 1024, 0)
ss = SpreadSpectrum(32, 1024, 0)
ss

SpreadSpectrum(number=32, length=1024, seed=0)

In [4]:
tx_bytes = np.random.randint(0, 255, size=1 << 12, dtype=np.uint8)
tx_bits = np.unpackbits(tx_bytes).reshape(-1, ss.number)
tx_bips = tx_bits.astype(np.int32) * 2 - 1

tx_bytes[:4], tx_bits[:4], tx_bips[:4]


(array([ 74, 116, 144,  70], dtype=uint8),
 array([[0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0,
         0, 0, 0, 1, 0, 0, 0, 1, 1, 0],
        [1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0,
         0, 0, 1, 1, 0, 1, 0, 1, 0, 0],
        [1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1,
         0, 1, 0, 0, 1, 0, 1, 0, 0, 1],
        [1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1,
         1, 0, 1, 1, 0, 0, 1, 1, 0, 0]], dtype=uint8),
 array([[-1,  1, -1, -1,  1, -1,  1, -1, -1,  1,  1,  1, -1,  1, -1, -1,
          1, -1, -1,  1, -1, -1, -1, -1, -1,  1, -1, -1, -1,  1,  1, -1],
        [ 1, -1,  1,  1,  1,  1, -1, -1,  1,  1, -1,  1, -1,  1,  1,  1,
         -1, -1, -1,  1, -1, -1, -1, -1,  1,  1, -1,  1, -1,  1, -1, -1],
        [ 1, -1,  1,  1,  1,  1, -1, -1,  1, -1,  1, -1, -1, -1, -1,  1,
         -1, -1, -1, -1,  1,  1, -1,  1, -1, -1,  1, -1,  1, -1, -1,  1],
        [ 1,  1,  1,  1,  1, -1, -1,  1, 

In [5]:
sm = ss.spreading_matrix()
sm.shape

(32, 1024)

In [6]:
tx_signal = tx_bips[0] @ sm
tx_signal.shape

(1024,)

In [7]:
ttx_signal = tx_bips @ sm
ttx_signal.shape, ttx_signal.size

((1024, 1024), 1048576)

In [8]:
tx_signal == ttx_signal[0]

array([ True,  True,  True, ...,  True,  True,  True], shape=(1024,))

In [9]:
rx_bips = sm @ tx_signal 
rx_bips

array([ -818.,  1010.,  -882.,  -970.,   990., -1266.,  1046.,  -930.,
       -1154.,  1046.,  1226.,  1038.,  -534.,   986.,  -890., -1350.,
        1042., -1218.,  -838.,   902.,  -886.,  -702.,  -978., -1010.,
        -874.,   942.,  -722., -1206.,  -878.,  1150.,   974., -1026.])

In [10]:
rrx_bips = ttx_signal @ sm.T
rrx_bips.shape, rrx_bips[0]

((1024, 32),
 array([ -818.,  1010.,  -882.,  -970.,   990., -1266.,  1046.,  -930.,
        -1154.,  1046.,  1226.,  1038.,  -534.,   986.,  -890., -1350.,
         1042., -1218.,  -838.,   902.,  -886.,  -702.,  -978., -1010.,
         -874.,   942.,  -722., -1206.,  -878.,  1150.,   974., -1026.]))

In [11]:
np.all((tx_bips > 0) == (rrx_bips > 0))

np.True_

In [12]:
rrx_bytes = np.packbits(rrx_bips > 0)
rrx_bytes.shape

(4096,)

In [13]:
np.all(rrx_bytes == tx_bytes), ss.length/ss.number*8, np.std(ttx_signal), np.std(rrx_bips)

(np.True_,
 256.0,
 np.float64(5.648547563149952),
 np.float64(1035.6273206322726))

In [14]:
ttx_signal.size/len(tx_bytes)

256.0

In [15]:
# all in one cell

for spreading_factor in [128, 256, 512, 1024, 2048, 4096]:
    for bits_per_symbol in [8, 16, 32, 64, 128, 256]:
        ss = SpreadSpectrum(bits_per_symbol, spreading_factor, 0)

        sm = ss.spreading_matrix()

        tx_bytes = np.random.randint(0, 255, size=1 << 16, dtype=np.uint8)
        tx_bits = np.unpackbits(tx_bytes).reshape(-1, ss.number)
        tx_bips = tx_bits.astype(np.int32) * 2 - 1
        ttx_signal = tx_bips @ sm

        rrx_signal = ttx_signal/np.std(ttx_signal)
        rrx_signal += np.random.normal(scale=100, size=ttx_signal.shape)

        rrx_bips = ttx_signal @ sm.T

        rrx_bips_ber = np.mean((tx_bips > 0) != (rrx_bips > 0))
        # print(f"{rrx_bips_ber=}")

        rrx_bytes = np.packbits(rrx_bips > 0)
        rrx_bytes_ber = np.mean(rrx_bytes != tx_bytes)
        # print(f"{rrx_bytes_ber=}")

        print(ss.length, ss.number, ss.length/ss.number*8, rrx_bips_ber)


128 8 128.0 0.0
128 16 64.0 0.0008373260498046875
128 32 32.0 0.02533721923828125
128 64 16.0 0.076873779296875
128 128 8.0 0.16010665893554688
128 256 4.0 0.24143409729003906
256 8 256.0 0.0
256 16 128.0 0.0
256 32 64.0 0.002170562744140625
256 64 32.0 0.024873733520507812
256 128 16.0 0.07765007019042969
256 256 8.0 0.15730667114257812
512 8 512.0 0.0
512 16 256.0 0.0
512 32 128.0 5.7220458984375e-06
512 64 64.0 0.0018634796142578125
512 128 32.0 0.020998001098632812
512 256 16.0 0.07690811157226562
1024 8 1024.0 0.0
1024 16 512.0 0.0
1024 32 256.0 0.0
1024 64 128.0 4.1961669921875e-05
1024 128 64.0 0.0021190643310546875
1024 256 32.0 0.021862030029296875
2048 8 2048.0 0.0
2048 16 1024.0 0.0
2048 32 512.0 0.0
2048 64 256.0 0.0
2048 128 128.0 2.288818359375e-05
2048 256 64.0 0.002288818359375
4096 8 4096.0 0.0
4096 16 2048.0 0.0
4096 32 1024.0 0.0
4096 64 512.0 0.0
4096 128 256.0 0.0
4096 256 128.0 2.6702880859375e-05
